In [21]:
import json
import os
import sys
import numpy as np
import cv2


In [9]:
with open('/home/dangnh36/datasets/ecg/processed/reference_keypoints.json', 'r') as f:
    ref_kpts = json.load(f)
ref_kpts

{'s_0_0_t': [609.251968505, 683.5332837332342],
 's_0_0_b': [609.251968505, 733.1333829334325],
 's_0_1_t': [1101.3779527559057, 683.5332837332342],
 's_0_1_b': [1101.3779527559057, 733.1333829334325],
 's_0_2_t': [1593.503937007874, 683.5332837332342],
 's_0_2_b': [1593.503937007874, 733.1333829334325],
 's_0_3_t': [2084.6456692913384, 683.5332837332342],
 's_0_3_b': [2084.6456692913384, 733.1333829334325],
 's_1_0_t': [609.2519685039371, 966.8666170665674],
 's_1_0_b': [609.2519685039371, 1016.4667162667657],
 's_1_1_t': [1101.3779527559057, 966.8666170665674],
 's_1_1_b': [1101.3779527559057, 1016.4667162667657],
 's_1_2_t': [1593.503937007874, 966.8666170665674],
 's_1_2_b': [1593.503937007874, 1016.4667162667657],
 's_1_3_t': [2084.6456692913384, 966.8666170665674],
 's_1_3_b': [2084.6456692913384, 1016.4667162667657],
 's_2_0_t': [609.2519685039371, 1250.199950399901],
 's_2_0_b': [609.2519685039371, 1299.800049600099],
 's_2_1_t': [1101.3779527559057, 1250.199950399901],
 's_2_1

In [10]:
ref_kpt_xys = np.array(list(ref_kpts.values()))
ref_kpt_names = list(ref_kpts.keys())
len(ref_kpt_names), ref_kpt_xys.shape

(57, (57, 2))

In [5]:
with open('/home/dangnh36/datasets/ecg/processed/imc_results.json', 'r') as f:
    imc_results = json.load(f)
len(imc_results)

977

In [6]:
all_sids = list(imc_results.keys())
len(all_sids)

977

```
2082219639 5
```

In [33]:
def batch_perspective_transform_2d(src_pts, M):
    B, N, _ = src_pts.shape
    _src_pts = np.ones((B, N, 3), dtype=np.float32)
    _src_pts[..., :2] = src_pts
    # (B,3,3) @ (B,3,N) -> (B, N, 3)
    dst_pts = np.matmul(M, _src_pts.transpose(0, 2, 1)).transpose(0, 2, 1)
    dst_pts[..., :2] = dst_pts[..., :2] / dst_pts[..., 2:3]
    return dst_pts[..., :2]


import cv2
import matplotlib.pyplot as plt
import numpy as np

def viz(bgr, kpt_xys, kpt_names=None, radius=5, thickness=2, font_scale=0.5):
    """
    Visualize 2D keypoints on a BGR image using matplotlib.
    
    Parameters
    ----------
    bgr : np.ndarray
        OpenCV image in BGR format.
    kpt_xys : np.ndarray or list
        Array of shape (N, 2) containing (x, y) coordinates.
    kpt_names : list[str] or None
        Optional list of keypoint names, length N.
    radius : int
        Radius of each point.
    thickness : int
        Outline thickness for each point.
    font_scale : float
        Text label scale.
    """
    img = bgr.copy()
    kpt_xys = np.array(kpt_xys, dtype=np.float32)

    for i, (x, y) in enumerate(kpt_xys):
        cv2.circle(img, (int(x), int(y)), radius, (0, 255, 0), -1, cv2.LINE_AA)
        # cv2.circle(img, (int(x), int(y)), radius, (0, 0, 0), thickness, cv2.LINE_AA)
        # if kpt_names is not None:
        #     name = str(kpt_names[i])
        #     cv2.putText(
        #         img, name,
        #         (int(x) + 6, int(y) - 6),
        #         cv2.FONT_HERSHEY_SIMPLEX,
        #         font_scale, (255, 255, 255), 1, cv2.LINE_AA
        #     )
        #     cv2.putText(
        #         img, name,
        #         (int(x) + 6, int(y) - 6),
        #         cv2.FONT_HERSHEY_SIMPLEX,
        #         font_scale, (0, 0, 0), 2, cv2.LINE_AA
        #     )

    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

In [ ]:
import random

IMG_DIR = '/home/dangnh36/datasets/ecg/raw/train/'

for sid in random.choices(all_sids, k=100):
    sid_imc_result = imc_results[sid]
    type_id = random.choice(list(sid_imc_result.keys()))
    imc_result = sid_imc_result[type_id]
    print(sid, type_id, imc_result)
    H = np.array(imc_result[3])
    H = np.linalg.pinv(H)
    rot_code = imc_result[1]

    dst_kpts = batch_perspective_transform_2d(ref_kpt_xys[None], H)[0]
    print(dst_kpts.shape)
    img_path = os.path.join(IMG_DIR, str(sid), f'{sid}-{int(type_id):04d}.png')
    img = cv2.imread(img_path)
    if rot_code is not None:
        img = cv2.rotate(img, rot_code)
    viz(img, dst_kpts, kpt_names=ref_kpt_names, radius=10, thickness=2, font_scale=0.5)